In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap

In [ ]:
# Reading well data for EA-36
slug_df = pd.read_csv('processed_EA-36.csv')

In [ ]:
slug_df

In [ ]:
#masking on few days only 
start_date = '2022-06-15 00:00:00'
end_date = '2022-06-17 06:00:00'
mask_ss = (slug_df[slug_df.columns[0]] > start_date) & (slug_df[slug_df.columns[0]] <= end_date)

slug_october = slug_df.loc[mask_ss]
print('lenght',len(slug_october))

fig, ts = plt.subplots(4,figsize=(10,10),sharex = True)

fig.suptitle('Slug dataset EA-36')
ts[2].set_xlabel('time')
ts[0].set_ylabel('FlowLine_Pressure')
ts[1].set_ylabel('TH_Pressure')
ts[2].set_ylabel('GL_Flowrate')
ts[3].set_ylabel('GL_Pressure')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_october[slug_october.columns[0]], slug_october["FlowLine_Pressure"])
ts[1].plot(slug_october[slug_october.columns[0]], slug_october["TH_Pressure"])
ts[2].plot(slug_october[slug_october.columns[0]], slug_october["GL_Flowrate"])
ts[3].plot(slug_october[slug_october.columns[0]], slug_october["GL_Pressure"])


In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

freq_units = 0.1/len(slug_october)
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('EA-36 THP stats')
data[0].set_xlabel('time (10s)')
data[0].set_ylabel('THP')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('THP')
data[2].set_ylabel('count')

#data[1].set_xlim(2,175)
#data[1].set_ylim(0,2160)
data[1].set_yscale('log')
data[1].set_xscale('log')

ss_THP = slug_october['TH_Pressure']

fft_ss = np.fft.rfft(ss_THP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_THP)
data[1].plot(power_spectrum_ss)
ss_THP.hist()

The time series does not show a particular periodicity...

I try anyway to do TDA and see if it picks up any periodicity the is not imediately visible to me. 

In [ ]:
max_time_delay = 100
max_embedding_dimension = 10
stride = 2

signal = slug_october['TH_Pressure']
print('length:', len(signal))

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 2

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)

No periodicity, it's just a shapeless blob. With periodicity I shaould have gotten some sort of defined shape.

Is it Gaussian (white) noise?

In [ ]:
from statsmodels.graphics.gofplots import qqplot
qqplot(signal, line='s')

No, it's not (and it was quite clear), probably some sort of colored noise. Not worth investigating more. 

Looking now at EA-10 as Andris suggested. 

In [ ]:
slug_10df = pd.read_csv('processed_EA-10.csv')

In [ ]:
slug_10df

In [ ]:
#masking on October 2022 only '2022-06-01 03:20:00') and Timestamp('2022-06-01 06:39:00'
start_date = '2022-06-01 03:20:00'
end_date = '2022-06-01 06:39:00'
mask_ss = (slug_10df[slug_10df.columns[0]] > start_date) & (slug_10df[slug_10df.columns[0]] <= end_date)

slug_october = slug_10df.loc[mask_ss]
print(len(slug_october))
#slug_october=slug_october[10000:]

fig, ts = plt.subplots(4,figsize=(10,10),sharex = True)

fig.suptitle('Slug dataset EA-36')
ts[2].set_xlabel('time')
ts[0].set_ylabel('FlowLine_Pressure')
ts[1].set_ylabel('TH_Pressure')
ts[2].set_ylabel('GL_Flowrate')
ts[3].set_ylabel('GL_Pressure')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_october[slug_october.columns[0]], slug_october["FlowLine_Pressure"])
ts[1].plot(slug_october[slug_october.columns[0]], slug_october["TH_Pressure"])
ts[2].plot(slug_october[slug_october.columns[0]], slug_october["GL_Flowrate"])
ts[3].plot(slug_october[slug_october.columns[0]], slug_october["GL_Pressure"])


In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

freq_units = 0.1/len(slug_october)
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('steady state BHP stats')
data[0].set_xlabel('time (min)')
data[0].set_ylabel('BHP')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP')
data[2].set_ylabel('count')

#data[1].set_xlim(2,175)
#data[1].set_ylim(0,2160)
data[1].set_yscale('log')
data[1].set_xscale('log')

ss_BHP = slug_october['TH_Pressure']

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
max_time_delay =20
max_embedding_dimension = 10
stride = 1

signal = slug_october['TH_Pressure']
print('length:', len(signal))

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)

Taking a larger window, just in case

In [ ]:
start_date = '2022-06-01 03:20:00'
end_date = '2022-06-01 08:99:00'
mask_ss = (slug_10df[slug_10df.columns[0]] > start_date) & (slug_10df[slug_10df.columns[0]] <= end_date)

slug_october = slug_10df.loc[mask_ss]

In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

freq_units = 0.1/len(slug_october)
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('steady state BHP stats')
data[0].set_xlabel('time (min)')
data[0].set_ylabel('BHP')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP')
data[2].set_ylabel('count')

#data[1].set_xlim(2,175)
#data[1].set_ylim(0,2160)
data[1].set_yscale('log')
data[1].set_xscale('log')

ss_BHP = slug_october['TH_Pressure']

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
max_time_delay =20
max_embedding_dimension = 10
stride = 1

signal = slug_october['TH_Pressure']
print('length:', len(signal))

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)

It's a bit pringles shaped but there is no apparent periodicity in the data that the embedding can catch. Probably I would need to play with the embedding parameters. 

In [ ]:
figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_october.index, y=slug_october['TH_Pressure'],  name="FHP")
figure_FHP.show()

In [ ]:
from scipy.interpolate import CubicSpline, PPoly, KroghInterpolator

xmin = 200
xmax = 400
some_step = 0.1
cs = CubicSpline(slug_october.index, slug_october['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)
plt.plot(x_range, cs(x_range), '+', label='Cubic Spline')

In [ ]:
max_time_delay =33
max_embedding_dimension = 15
stride = 2

signal = cs(x_range)
print('length:', len(signal))

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 2

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_ss_BHP[None,:,:])

In [ ]:
# Computing the Entropy of the homology classes (the lower the more ordered is stuff)

PE_FHP = PersistenceEntropy()
features = PE_FHP.fit_transform(PerDiagram)
features 

In [ ]:
xmin = 1
xmax = 2000
some_step = 0.1
cs = CubicSpline(slug_10df.index, slug_10df['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)
plt.plot(x_range, cs(x_range), '+', label='Cubic Spline')

In [ ]:
max_time_delay =33
max_embedding_dimension = 15
stride = 1

signal = cs(x_range)[:1000]
print('length:', len(signal))

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_ss_BHP = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_ss_BHP_pca = pca.fit_transform(y_ss_BHP)

plot_point_cloud(y_ss_BHP_pca)


In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_ss_BHP[None,:,:])

In [ ]:
PE_FHP = PersistenceEntropy()
features = PE_FHP.fit_transform(PerDiagram)
features 